# **Visualization of CNNs: Grad-CAM**
* **Objective**: Convolutional Neural Networks are widely used on computer vision. They are powerful for processing grid-like data. However we hardly know how and why they work, due to the lack of decomposability into individually intuitive components. In this assignment, we use Grad-CAM, which highlights the regions of the input image that were important for the neural network prediction.


* NB: if `PIL` is not installed, try `conda install pillow`.
* Computations are light enough to be done on CPU.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models, datasets, transforms
import matplotlib.pyplot as plt
import pickle
import urllib.request

import numpy as np
import math
from PIL import Image

%matplotlib inline

## Download the Model
We provide you with a model `DenseNet-121`, already pretrained on the `ImageNet` classification dataset.
* **ImageNet**: A large dataset of photographs with 1 000 classes.
* **DenseNet-121**: A deep architecture for image classification (https://arxiv.org/abs/1608.06993)

In [ ]:
model = models.densenet121(pretrained=True)
model.eval() # set the model to evaluation model
layers = model.features

In [ ]:
model = models.resnext50_32x4d(pretrained=True)
model.eval()
layers = model.layer4

In [ ]:
classes = pickle.load(urllib.request.urlopen('https://gist.githubusercontent.com/yrevar/6135f1bd8dcf2e0cc683/raw/d133d61a09d7e5a3b36b8c111a8dd5c4b5d560ee/imagenet1000_clsid_to_human.pkl'))

##classes is a dictionary with the name of each class 
print(classes)

## Input Images
We provide you with 20 images from ImageNet (download link on the webpage of the course or download directly using the following command line,).<br>
In order to use the pretrained model resnet34, the input image should be normalized using `mean = [0.485, 0.456, 0.406]`, and `std = [0.229, 0.224, 0.225]`, and be resized as `(224, 224)`.

In [ ]:
mean = [0.485, 0.456, 0.406]
std  = [0.229, 0.224, 0.225]

normalize = transforms.Normalize(mean=mean,
                                     std=std)

inv_normalize = transforms.Normalize(
    mean=[-m/s for m, s in zip(mean, std)],
    std=[1/s for s in std]
)

transform = transforms.Compose([
                transforms.Resize(256), 
                transforms.CenterCrop(224), # resize the image to 224x224
                transforms.ToTensor(), # convert numpy.array to tensor
                normalize]
            )

def preprocess_image(dir_path):
    
    dataset = datasets.ImageFolder(dir_path, transform) #normalize the tensor

    return (dataset)

In [ ]:
import os
if not os.path.exists("data"):
    os.mkdir("data")
if not os.path.exists("data/TP2_images"):
    os.mkdir("data/TP2_images")
    !cd data/TP2_images && wget "https://www.lri.fr/~gcharpia/dlip/2026/TP2/TP2_images.zip" && unzip TP2_images.zip

dir_path = "data/" 
dataset = preprocess_image(dir_path)

In [ ]:
# show the original image 
index = 15  # for comparison later, revert to ``15''
input_image = Image.open(dataset.imgs[index][0]).convert('RGB')
plt.imshow(input_image)
plt.axis('off')
plt.show()

In [ ]:
input_image = dataset[index][0].view(1, 3, 224, 224)
# input_image = dataset[index][0].view(1, 3, 224, 224)
# output = densenet121(input_image)
output = model(input_image)
values, indices = torch.topk(output, 4)
print("Top 3-classes:", indices[0].numpy(), [classes[x] for x in indices[0].numpy()])
print("Raw class scores:", values[0].detach().numpy())

# Grad-CAM 
* **Overview:** Given an image, and a category (‘tiger cat’) as input, we forward-propagate the image through the model to obtain the `raw class scores` before softmax. The gradients are set to zero for all classes except the desired class (tiger cat), which is set to 1. This signal is then backpropagated to the `rectified convolutional feature map` of interest, where we can compute the coarse Grad-CAM localization (blue heatmap).


* **To Do**: Define your own function Grad_CAM to achieve the visualization of the given images. For each image, choose the top-3 possible labels as the desired classes. Compare the heatmaps of the three classes, and conclude. 

More precisely, you should provide a function: `show_grad_cam(image: torch.tensor) -> None` that displays something like this:

![output_seal.png](output_seal.png)

where the heatmap will be correct (here it is just an example) and the first 3 classes are the top-3 predicted classes and the last is the least probable class according to the model.

* **Comment your code**: Your code should be easy to read and follow. Please comment your code, try to use the NumPy Style Python docstrings for your functions.

* **To be submitted within 2 weeks**: this notebook, **cleaned** (i.e. without results, for file size reasons: `menu > kernel > restart and clean`), in a state ready to be executed (with or without GPU) (if one just presses 'Enter' till the end, one should obtain all the results for all images) with a few comments at the end. No additional report, just the notebook!


* **Hints**: 
 + We need to record the output and grad_output of the feature maps to achieve Grad-CAM. In pytorch, the function `Hook` is defined for this purpose. Read the tutorial of [hook](https://pytorch.org/tutorials/beginner/former_torchies/nnft_tutorial.html#forward-and-backward-function-hooks) carefully.
 + More on [autograd](https://pytorch.org/tutorials/beginner/basics/autogradqs_tutorial.html) and [hooks](https://pytorch.org/tutorials/beginner/former_torchies/nnft_tutorial.html#forward-and-backward-function-hooks)
 + The pretrained model densenet doesn't have an activation function after its last layer, the output is indeed the `raw class scores`, you can use them directly. 
 + Your heatmap will have the same size as the feature map. You need to scale up the heatmap to the resized image (224x224, not the original one, before the normalization) for better observation purposes. The function [`torch.nn.functional.interpolate`](https://pytorch.org/docs/stable/nn.functional.html?highlight=interpolate#torch.nn.functional.interpolate) may help.  
 + Here is the link to the paper: [Grad-CAM: Visual Explanations from Deep Networks via Gradient-based Localization](https://arxiv.org/pdf/1610.02391.pdf)

Class: ‘pug, pug-dog’ | Class: ‘tabby, tabby cat’  

 
![alt](https://raw.githubusercontent.com/jacobgil/pytorch-grad-cam/master/examples/dog.jpg)| ![alt](https://raw.githubusercontent.com/jacobgil/pytorch-grad-cam/master/examples/cat.jpg)

## Part 1: Grad-CAM implementation

In [ ]:

class Hook:
    """
    Forward and backward hook for a target nn.Module to capture feature maps and gradients.
    
    Attributes:
        features (torch.Tensor): Activations from the forward pass.
        grad (torch.Tensor): Gradients w.r.t. the activations.
    """

    def __init__(self, target: nn.Module):
        self.forward_handle = target.register_forward_hook(self.forward)
        self.backward_handle = None
        self.features: torch.Tensor = None
        self.grad: torch.Tensor = None

    def forward(self, module, input, output: torch.Tensor):
        """
        Forward hook to store the output features and attach a gradient hook.
        """
        self.features = output
        self.backward_handle = output.register_hook(self.backward)

    def backward(self, grad: torch.Tensor):
        """
        Backward hook to store the gradient of the output.
        """

        self.grad = grad.clone()[0] # clone seems to be required for densenet121

    def remove(self):
        """
        Remove the registered hooks.
        """
        self.forward_handle.remove()
        if self.backward_handle is not None:
            self.backward_handle.remove()


def get_heat_map(logits: torch.Tensor, hook: Hook, idx: torch.Tensor) -> torch.Tensor:
    """
    Compute the Grad-CAM heat map for a single class index.

    Args:
        logits (torch.Tensor): Output logits from the model.
        hook (Hook): Hook object containing features and gradients.
        idx (torch.Tensor): Index of the target class.

    Returns:
        torch.Tensor: The class-discriminative heat map (ReLU applied).
    """
    logit = logits[0, idx]
    logit.backward(retain_graph=True)
    alphas = hook.grad.mean(dim=(-2, -1), keepdim=True)
    heat_map = torch.sum(alphas * hook.features, dim=1, keepdim=True)
    return F.relu(heat_map)


def match_size(target: torch.Tensor, x: torch.Tensor) -> torch.Tensor:
    """
    Upsample tensor x to match the spatial size of target.

    Args:
        target (torch.Tensor): Reference tensor whose size to match.
        x (torch.Tensor): Tensor to resize.

    Returns:
        torch.Tensor: Resized tensor.
    """
    size = target.shape[-2:]
    return F.interpolate(x, size=size, mode='bilinear')

def grad_cam(model: nn.Module, layer: nn.Module, input_image: torch.Tensor, topk: int) -> list:
    """
    Compute Grad-CAM heat maps for the top-k classes.

    Args:
        model (nn.Module): The neural network model.
        layer (nn.Module): The target convolutional layer to hook.
        input_image (torch.Tensor): Input image tensor (1, C, H, W).
        topk (int): Number of top classes to compute Grad-CAM for.

    Returns:
        List[Tuple[torch.Tensor, torch.Tensor, str]]: List of tuples (heat_map, probability, class_name)
    """
    model.zero_grad(set_to_none=True)
    hook = Hook(layer)
    logits = model(input_image)
    probs = F.softmax(logits, dim=1)

    heat_maps = []

    for idx in torch.argsort(logits, dim=1)[0][-topk:]:
        heat_map = get_heat_map(logits, hook, idx)
        heat_map = match_size(input_image, heat_map)
        prob = probs[0, idx]
        cat = classes[idx.cpu().detach().item()]
        heat_maps.append((heat_map, prob, cat))

    heat_maps.reverse()
    hook.remove()
    return heat_maps

In [ ]:
def show_grad_cam(image: torch.Tensor, model: nn.Module, layer: nn.Module, topk: int) -> None:

    heat_maps = grad_cam(model, layer, image, topk)

    n = len(heat_maps) + 1

    cols = min(4, n)
    rows = math.ceil(n / cols)

    fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 4 * rows))
    axes = axes.flatten() if n > 1 else [axes]

    img = inv_normalize(image).squeeze().permute((1, 2, 0)).cpu()

    axes[0].imshow(img)
    axes[0].set_title("Original image")
    axes[0].axis("off")

    for ax, (heat_map, prob, cat) in zip(axes[1:], heat_maps):
        ax.imshow(img)
        ax.imshow(
            heat_map[0, 0, ...].detach().cpu(),
            alpha=0.5,
            cmap="jet"
        )
        ax.set_title(f"{cat} : {prob:.2f}")
        ax.axis("off")

    for ax in axes[len(heat_maps) + 1:]:
        ax.axis("off")

    plt.tight_layout()
    plt.show()


## Part 2: Try it on a few (1 to 3) images and comment

In [ ]:
layer = layers[-1]
topk = 3

In [ ]:
show_grad_cam(input_image, model, layer, topk)

Given that the results for this image are identical to the example provided, we can assume the implementation is correct.

In [ ]:
idx = 8
image = dataset[idx][0].view(1, 3, 224, 224)
show_grad_cam(image, model, layer, topk)

In [ ]:
idx = 10
image = dataset[idx][0].view(1, 3, 224, 224)
show_grad_cam(image, model, layer, topk)

The Grad-CAM explanation is always focused on the right specific region of the image for the predicted class. Furthermore, we can observe that Grad-CAM can distinguish between different instances of the same object in a single image: for the sorrel horse class, the heatmap is mainly concentrated on one instance (the larger horse).

We also notice that when the model is very confident in its top prediction, the explanations for the second and third predictions still highlight the region corresponding to the true class. This could be explained by the fact that if the sea lion were not present in the image, those alternative classes would have a higher probability. However, this is exactly the kind of behaviour that the ReLU step in Grad-CAM is meant to reduce, since it removes negative evidence and keeps only features that positively contribute to the target class.

For the mink prediction, we can see that the model is not very confident when distinguishing it from a weasel. These two animals can often be differentiated by their size, but the network does not seem to rely on that cue. Instead, the difference between the mink and weasel heatmaps is mainly located around the animal’s face, suggesting that the network is using facial features to make the distinction.

## Part 3: Try GradCAM on others convolutional layers, describe and comment the results

In [ ]:
topk = 3

In [ ]:
layer = layers[-1]
topk = 3
show_grad_cam(input_image, model, layer, topk)

In [ ]:
layer = layers[-2]
topk = 3
show_grad_cam(input_image, model, layer, topk)

In [ ]:
layer = layers[-3]
show_grad_cam(input_image, model, layer, topk)

As expected, Grad-CAM produces better results when applied to deeper feature maps. In particular, the localization of the heatmap for the sea lion is much more accurate when using the last convolutional layer compared to earlier layers. This can be explained by the fact that deeper layers contain more high-level semantic information, since semantic features progressively emerge across successive convolutional layers, while still preserving spatial localization. In contrast, this spatial information is largely lost in the classification head, typically due to operations such as global pooling and fully connected layers which is why rwe run the algorithm on the last convolution layer.


## Part 4: Try GradCAM on the images `9928031928X{1-4}.png` , describe and comment the results. What can you conclude ?

In [ ]:
img_path = "data/TP2_images/TP2_images/9928031928X"
image = dataset[15][0].view(1, 3, 224, 224)

for idx in range(1, 5):
    img = Image.open(img_path + f'{idx}.png').convert("RGB")  # make sure it's RGB
    img = transform(img)
    img = img.unsqueeze(0)
    layer = layers[-1]
    
    value_range = img.max() - img.min()
    error = torch.mean(torch.abs((image - img)/value_range) * 100)
    print(f'error : {error.item() : .2f} %')
    
    show_grad_cam(img, model, layer, topk)
    

These images appear to be identical, except for some added noise corresponding to an estimated 12% change in pixel values when considering the range of the perturbation (here we assume that the image with index 15 is the original image). We can see that the Grad-CAM heatmap remains concentrated on the sea lion and still provides a reliable explanation for the classifier’s decision. For the lower-confidence classes, the explanations do not change much either, except that the classifier becomes less confident in its predictions.

This result is consistent with what is reported in the paper, which claims that Grad-CAM visualizations are relatively robust to noise and even to adversarial perturbations.

## Part 5: What are the principal contributions of GradCAM (the answer is in the paper) ?

The main contribution of Grad-CAM is generalizing the CAM method so that it can be applied to any convolutional neural network for classification. While CAM is limited to architectures with global average pooling followed by a fully connected classification layer (as it uses the weight of the classification head instead if the gradient to average the features), Grad-CAM does not require any specific structure after the last convolutional layer.

Grad-CAM also introduces an improvement called Guided Grad-CAM, which combines Grad-CAM with guided backpropagation to produce high-resolution, class-discriminative visualizations.

In the context of image classification, these visualizations provide several benefits:
- They offer insights into model failure modes, showing that seemingly unreasonable predictions often have understandable explanations.
- They outperform previous methods on the ILSVRC-15 weakly-supervised localization task.
- They are robust to adversarial perturbations.
- They are more faithful to the underlying model, accurately reflecting which regions influence the prediction.
- They help improve model generalization by revealing dataset biases.

## Bonus : What are the main differences between ResNet and ResNeXt ?

The main difference between ResNeXt and ResNet is that ResNeXt introduces groups, meaning each residual block is split into several parallel paths. The input is processed by these paths in parallel, and their outputs are then combined. This allows the network to learn a more diverse set of features and increases its representational power, often improving performance without drastically increasing complexity. In comparison, a ResNet block can be seen as a special case of ResNeXt with only one path.

In practice ResNeXt can achieve similar or better result than ResNet with less parameters.